# Walmart M5 Demand Forecasting — Microsoft Fabric

End-to-end pipeline on one lakehouse (medallion layout):

| Layer | Table | What it holds |
|---|---|---|
| Bronze | `bronze_sales`, `bronze_calendar`, `bronze_prices` | Raw M5 CSVs from Kaggle as Delta |
| Silver | `silver_features` | Daily sales per item (long format) + calendar, price and lag features |
| Gold | `gold_forecast`, `gold_model_metrics` | Forecasts vs actuals and model metrics for Power BI |

Model: **LightGBM** (Tweedie objective, early stopping on a 28-day holdout), tracked and registered with **MLflow** (Fabric *Experiment* + *ML model* items).

**Forecast design:** every feature is lagged by **at least 28 days**, so one model can forecast the full 28-day horizon
without using any sales it would not know yet (no leakage).

**Before running:** attach a lakehouse as default. Either upload the M5 CSVs (`calendar.csv`, `sales_train_evaluation.csv`, `sell_prices.csv`)
to its `Files/` root, or upload `kaggle.json` there and the notebook downloads them (requires accepting the M5 competition rules on Kaggle).

## 1. Configuration

In [ ]:
# Scope: start with one store to keep it fast on a trial capacity; add more stores to scale up.
STORES = ["CA_1"]               # e.g. ["CA_1", "TX_1", "WI_1"] or all 10 stores

TRAIN_START_DAY = 1100          # d_1100 = 2014-02-02 — train on ~2 years of history
VALID_START_DAY = 1914          # last 28 known days (d_1914..d_1941) are the holdout
LAST_KNOWN_DAY  = 1941          # last day with actual sales in M5
HORIZON_END_DAY = 1969          # forecast the next 28 days (d_1942..d_1969)
NUM_BOOST_ROUND = 800           # upper limit; early stopping picks the best round

EXPERIMENT = "walmart_m5_forecast"
MODEL_NAME = "walmart_m5_lgbm"

FILES     = "/lakehouse/default/Files"   # local file API path of the default lakehouse
RAW_DIR   = FILES                         # M5 CSVs live in the Files/ root of the lakehouse
SPARK_RAW = "Files"                       # same folder, as Spark sees it

## 2. Download M5 data from Kaggle (skipped if already in the lakehouse)

In [ ]:
import os, json, base64, zipfile, requests

needed = ["calendar.csv", "sales_train_evaluation.csv", "sell_prices.csv"]

if all(os.path.exists(f"{RAW_DIR}/{f}") for f in needed):
    print("M5 files already in the lakehouse — skipping download")
else:
    with open(f"{FILES}/kaggle.json") as f:
        tok = json.load(f)
    auth = base64.b64encode(f"{tok['username']}:{tok['key']}".encode()).decode()
    url = "https://www.kaggle.com/api/v1/competitions/data/download-all/m5-forecasting-accuracy"
    r = requests.get(url, headers={"Authorization": f"Basic {auth}"}, stream=True, timeout=600)
    if r.status_code != 200:
        raise RuntimeError(
            f"Kaggle download failed ({r.status_code}): {r.text[:300]}\n"
            "Check kaggle.json and that you accepted the M5 competition rules on Kaggle."
        )
    zip_path = f"{FILES}/m5.zip"
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
    os.makedirs(RAW_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RAW_DIR)
    os.remove(zip_path)
    print("Downloaded:", sorted(os.listdir(RAW_DIR)))

## 3. Bronze — raw CSVs to Delta tables

In [ ]:
def load_csv(name):
    return spark.read.csv(f"{SPARK_RAW}/{name}", header=True, inferSchema=True)

load_csv("sales_train_evaluation.csv").write.format("delta").mode("overwrite").saveAsTable("bronze_sales")
load_csv("calendar.csv").write.format("delta").mode("overwrite").saveAsTable("bronze_calendar")
load_csv("sell_prices.csv").write.format("delta").mode("overwrite").saveAsTable("bronze_prices")

for t in ["bronze_sales", "bronze_calendar", "bronze_prices"]:
    print(t, spark.table(t).count())

## 4. Silver — long format + features

* Wide `d_1 … d_1941` columns are unpivoted to one row per item per day.
* Rows for the 28 future days are added (sales unknown) so the same features can be built for them.
* Days before an item was first sold (no price) are dropped.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
d_num = lambda c: F.regexp_extract(F.col(c), r"d_(\d+)", 1).cast("int")

sales_wide = spark.table("bronze_sales").filter(F.col("store_id").isin(STORES))
day_cols = [c for c in sales_wide.columns if c.startswith("d_")]
stack_expr = f"stack({len(day_cols)}, " + ", ".join(f"'{c}', `{c}`" for c in day_cols) + ") as (d, sales)"

history = (sales_wide.select(*id_cols, F.expr(stack_expr))
           .withColumn("d_num", d_num("d"))
           .withColumn("sales", F.col("sales").cast("double")))

cal = (spark.table("bronze_calendar")
       .withColumn("d_num", d_num("d"))
       .withColumn("date", F.to_date("date")))

future = (sales_wide.select(*id_cols)
          .crossJoin(cal.filter((F.col("d_num") > LAST_KNOWN_DAY) & (F.col("d_num") <= HORIZON_END_DAY)).select("d", "d_num"))
          .withColumn("sales", F.lit(None).cast("double")))

grid = history.unionByName(future)

df = (grid
      .join(cal.select("d_num", "date", "wm_yr_wk", "wday", "month", "year",
                       "event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI"), "d_num", "left")
      .join(spark.table("bronze_prices"), ["store_id", "item_id", "wm_yr_wk"], "left")
      .filter(F.col("sell_price").isNotNull())
      .withColumn("snap", F.when(F.col("state_id") == "CA", F.col("snap_CA"))
                           .when(F.col("state_id") == "TX", F.col("snap_TX"))
                           .otherwise(F.col("snap_WI")))
      .drop("snap_CA", "snap_TX", "snap_WI"))

# All sales features are shifted by >= 28 days, so they are known for every day of the 28-day horizon.
w = Window.partitionBy("id").orderBy("d_num")
df = (df
      .withColumn("lag_28", F.lag("sales", 28).over(w))
      .withColumn("lag_35", F.lag("sales", 35).over(w))
      .withColumn("lag_42", F.lag("sales", 42).over(w))
      .withColumn("rmean_7_28",  F.avg("sales").over(w.rowsBetween(-34, -28)))
      .withColumn("rmean_28_28", F.avg("sales").over(w.rowsBetween(-55, -28)))
      .withColumn("rstd_28_28",  F.stddev("sales").over(w.rowsBetween(-55, -28)))
      .withColumn("rmean_90_28", F.avg("sales").over(w.rowsBetween(-117, -28)))
      .withColumn("price_rel_max", F.col("sell_price") / F.max("sell_price").over(Window.partitionBy("id")))
      .withColumn("price_change_wk", F.col("sell_price") / F.lag("sell_price", 7).over(w) - 1)
      .withColumn("day_of_month", F.dayofmonth("date")))

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_features")
print("silver_features rows:", spark.table("silver_features").count())

## 5. Train / validation / future split

In [ ]:
import numpy as np
import pandas as pd

CAT_FEATURES = ["item_id", "dept_id", "cat_id", "store_id", "event_name_1", "event_type_1"]
NUM_FEATURES = ["wday", "month", "day_of_month", "snap", "sell_price", "price_rel_max", "price_change_wk",
                "lag_28", "lag_35", "lag_42", "rmean_7_28", "rmean_28_28", "rstd_28_28", "rmean_90_28"]
FEATURES = CAT_FEATURES + NUM_FEATURES

pdf = spark.table("silver_features").filter(F.col("d_num") >= TRAIN_START_DAY).toPandas()
pdf["date"] = pd.to_datetime(pdf["date"])
for c in CAT_FEATURES:
    pdf[c] = pdf[c].fillna("none").astype("category")
for c in NUM_FEATURES:
    pdf[c] = pdf[c].astype("float64")

train  = pdf[pdf["d_num"] <  VALID_START_DAY]
valid  = pdf[(pdf["d_num"] >= VALID_START_DAY) & (pdf["d_num"] <= LAST_KNOWN_DAY)].copy()
known  = pdf[pdf["d_num"] <= LAST_KNOWN_DAY]
future = pdf[pdf["d_num"] >  LAST_KNOWN_DAY].copy()

print(f"train {train.shape}  valid {valid.shape}  future {future.shape}")

## 6. Train and evaluate (tracked in MLflow)

Baseline to beat: the item's average daily sales over the 28 days before the horizon (`rmean_28_28`).

In [ ]:
import lightgbm as lgb
import mlflow, mlflow.lightgbm
from sklearn.metrics import mean_squared_error, mean_absolute_error

mlflow.autolog(disable=True)          # we log explicitly
mlflow.set_experiment(EXPERIMENT)

PARAMS = {
    "objective": "tweedie", "tweedie_variance_power": 1.1,   # suits many zero-sales days
    "metric": "rmse", "learning_rate": 0.05, "num_leaves": 128, "min_data_in_leaf": 100,
    "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 0.1,
    "verbose": -1, "seed": 42,
}

def metrics(y, p, dates, prefix):
    daily = pd.DataFrame({"date": dates, "y": y, "p": p}).groupby("date").sum()
    return {
        f"{prefix}_rmse": float(np.sqrt(mean_squared_error(y, p))),
        f"{prefix}_mae":  float(mean_absolute_error(y, p)),
        f"{prefix}_daily_total_mape": float((abs(daily.p - daily.y) / daily.y).mean() * 100),
    }

with mlflow.start_run(run_name="lgbm_validation") as run:
    dtrain = lgb.Dataset(train[FEATURES], label=train["sales"], categorical_feature=CAT_FEATURES)
    dvalid = lgb.Dataset(valid[FEATURES], label=valid["sales"], reference=dtrain)
    val_model = lgb.train(PARAMS, dtrain, NUM_BOOST_ROUND, valid_sets=[dvalid],
                          callbacks=[lgb.log_evaluation(50), lgb.early_stopping(50)])

    valid["forecast"] = np.clip(val_model.predict(valid[FEATURES]), 0, None)
    valid["baseline"] = valid["rmean_28_28"].fillna(0)

    m = {**metrics(valid["sales"], valid["forecast"], valid["date"], "lgbm"),
         **metrics(valid["sales"], valid["baseline"], valid["date"], "baseline")}
    mlflow.log_params({**PARAMS, "num_boost_round": NUM_BOOST_ROUND, "stores": ",".join(STORES),
                       "train_start_day": TRAIN_START_DAY})
    mlflow.log_metrics(m)
    validation_run_id = run.info.run_id

pd.Series(m).round(3)

## 7. Final model — retrain on all known days and register

The registered model is stored by MLflow in LightGBM's native text format inside OneLake
and appears as the **ML model** item `walmart_m5_lgbm` in the workspace.
A portable copy (`Files/models/walmart_m5_lgbm.txt`) is also saved — it loads anywhere with
`lgb.Booster(model_file=...)`, e.g. for the AWS version of this project.

In [ ]:
from mlflow.models import infer_signature

with mlflow.start_run(run_name="lgbm_final") as run:
    dall = lgb.Dataset(known[FEATURES], label=known["sales"], categorical_feature=CAT_FEATURES)
    final_model = lgb.train(PARAMS, dall, val_model.best_iteration)   # rounds chosen by early stopping

    sample = known[FEATURES].head(5)
    info = mlflow.lightgbm.log_model(
        final_model, artifact_path="model",
        registered_model_name=MODEL_NAME,
        signature=infer_signature(sample, final_model.predict(sample)),
    )
    mlflow.log_params({**PARAMS, "num_boost_round": NUM_BOOST_ROUND, "stores": ",".join(STORES),
                       "trained_through_day": LAST_KNOWN_DAY})
    mlflow.log_metrics({**m, "best_iteration": val_model.best_iteration})   # holdout scores of the validation run
    mlflow.set_tag("validation_run_id", validation_run_id)

model_version = str(getattr(info, "registered_model_version", None) or
                    max(int(v.version) for v in mlflow.MlflowClient().search_model_versions(f"name='{MODEL_NAME}'")))

os.makedirs(f"{FILES}/models", exist_ok=True)
final_model.save_model(f"{FILES}/models/{MODEL_NAME}.txt")
print(f"Registered {MODEL_NAME} v{model_version}; portable copy saved to Files/models/{MODEL_NAME}.txt")

## 8. Gold — forecasts and metrics for Power BI

In [ ]:
future["forecast"] = np.clip(final_model.predict(future[FEATURES]), 0, None)
future["baseline"] = np.nan

out_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "date", "d_num",
            "sales", "forecast", "baseline", "sell_price", "event_name_1"]
gold = pd.concat([valid[out_cols].assign(split="validation", model_version=f"validation_run"),
                  future[out_cols].assign(split="future", model_version=f"v{model_version}")])
gold = gold.rename(columns={"sales": "actual_sales"})
for c in CAT_FEATURES[:4] + ["event_name_1"]:
    gold[c] = gold[c].astype(str).replace("none", None)
gold["date"] = gold["date"].dt.date

gold_sdf = spark.createDataFrame(gold)
for c in ["actual_sales", "forecast", "baseline", "sell_price"]:      # pandas NaN -> real NULL for Power BI
    gold_sdf = gold_sdf.withColumn(c, F.when(F.isnan(c), None).otherwise(F.col(c)))
gold_sdf.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_forecast")

metrics_pdf = pd.DataFrame([
    {"model": name, "rmse": m[f"{name}_rmse"], "mae": m[f"{name}_mae"],
     "daily_total_mape": m[f"{name}_daily_total_mape"], "stores": ",".join(STORES),
     "model_version": f"v{model_version}"}
    for name in ["lgbm", "baseline"]])
spark.createDataFrame(metrics_pdf).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_model_metrics")

print("gold_forecast rows:", spark.table("gold_forecast").count())
display(spark.table("gold_model_metrics"))

## 9. Actual vs forecast (daily total)

In [ ]:
import matplotlib.pyplot as plt

hist = known[known["d_num"] >= VALID_START_DAY - 56].groupby("date")["sales"].sum()
val  = valid.groupby("date")["forecast"].sum()
fut  = future.groupby("date")["forecast"].sum()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(hist.index, hist.values, label="Actual", color="#333")
ax.plot(val.index,  val.values,  label="Forecast (holdout)", color="#1f77b4")
ax.plot(fut.index,  fut.values,  label="Forecast (next 28 days)", color="#ff7f0e", linestyle="--")
ax.axvline(pd.Timestamp(val.index.min()), color="grey", linewidth=0.8)
ax.set_title(f"Daily unit sales — stores {', '.join(STORES)}")
ax.set_ylabel("Units"); ax.legend(); plt.tight_layout(); plt.show()

## 10. Next: Power BI

Create a semantic model on `gold_forecast` + `gold_model_metrics` (Direct Lake) and build the report:
actual vs forecast over time, accuracy by category/department, and the 28-day outlook.

To score new data later without retraining:
```python
model = mlflow.lightgbm.load_model(f"models:/{MODEL_NAME}/{model_version}")
```